# infertility-prediction

원본 분석 코드의 공개용 복사본입니다. 원본 셀 번호는 메타데이터에 기록했습니다. 교육용 문제 설명·개별 기록 출력·계정/개인 경로를 제거했습니다. 데이터는 별도 승인 후 확보해야 합니다. 먼저 프로젝트 README를 읽으세요.

In [ ]:
from pathlib import Path
import os
import platform
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if PROJECT_ROOT.name != "infertility-prediction":
    raise RuntimeError("작업 디렉터리를 projects/infertility-prediction 또는 그 notebooks 폴더로 설정하세요.")
DATA_DIR = Path(os.environ.get("HEALTHCARE_DATA_DIR", str(PROJECT_ROOT / "data")))
ALLOW_TRAINING = False  # 학습 셀 실행을 원할 때만 직접 True로 변경


## 사용자 정의 전처리 클래스를 만들기 위한 기본 클래스

원본 코드 셀 3. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm

# 사용자 정의 전처리 클래스를 만들기 위한 기본 클래스
from sklearn.base import BaseEstimator, TransformerMixin

# 수치형과 범주형 변수에 서로 다른 처리를 적용하는 도구
from sklearn.compose import ColumnTransformer

# 범주형 인코딩은 학습 데이터의 범주 사전만 사용하는 방식으로 직접 구현합니다. (2장 참조)

# 여러 전처리 및 모델 학습 단계를 하나로 연결하는 도구
from sklearn.pipeline import Pipeline

# 데이터 분할 및 교차 검증에 사용하는 도구
from sklearn.model_selection import (
    StratifiedKFold,   # 타깃 클래스 비율을 유지하면서 K개로 분할
    cross_validate,    # 교차 검증을 실행하고 성능을 계산
    train_test_split   # 학습 데이터와 검증 데이터를 분리
)

# 모델 성능 평가에 사용하는 지표
from sklearn.metrics import (
    roc_auc_score,           # ROC-AUC 계산
    accuracy_score,          # 정확도 계산
    precision_score,         # 정밀도 계산
    recall_score,            # 재현율 계산
    f1_score,                # F1-score 계산
    classification_report    # 분류 성능 종합 보고서
)

# XGBoost 이진 분류 모델
from xgboost import XGBClassifier

# 난수 고정: 실행할 때마다 최대한 동일한 결과가 나오도록 설정
RANDOM_STATE = 42


## 분석 단계 7

원본 코드 셀 7. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
PROJECT_PATH = PROJECT_ROOT / 'outputs'
PROJECT_PATH.mkdir(exist_ok=True)


## 분석 단계 9

원본 코드 셀 9. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
DATA_PATH = DATA_DIR
TRAIN_DATA_PATH = DATA_PATH / 'train.csv'
TEST_DATA_PATH = DATA_PATH / 'test.csv'


## CSV 파일 불러오기

원본 코드 셀 14. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# CSV 파일 불러오기
# UTF-8-SIG 인코딩을 사용하여 한글 및 BOM(Byte Order Mark) 처리
train = pd.read_csv(TRAIN_DATA_PATH, encoding = 'utf-8-sig')
test = pd.read_csv(TEST_DATA_PATH, encoding = 'utf-8-sig')

# 데이터셋 크기 확인
# shape: (행 개수, 열 개수) 형태로 출력
print("Train:", train.shape)
print("Test:", test.shape)

# 상위 5개 train 데이터 출력
display(train.head())


## 상위 5개 test 데이터 출력

원본 코드 셀 15. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# 상위 5개 test 데이터 출력
display(test.head())


## ==========================================

원본 코드 셀 18. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# ==========================================
# Train 데이터 결측치 확인
# ==========================================
# 대회 규칙에 따라 Test 데이터는 분석하지 않고 Train 데이터만 분석합니다.

missing_info = pd.DataFrame({
    '결측치 개수': train.isnull().sum(),
    '결측치 비율 (%)': train.isnull().mean() * 100
})

# 결측치가 있는 컬럼만 추출하고 결측치 비율 기준으로 정렬
missing_info = missing_info[missing_info['결측치 개수'] > 0].sort_values(
    by='결측치 비율 (%)',
    ascending=False
)

display(missing_info.round(4))
print(f'Train 결측치 컬럼 수: {len(missing_info)}개')


## ==========================================

원본 코드 셀 19. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# ==========================================
# DI vs IVF 컬럼별 결측치 비율 비교
# ==========================================

# 시술 유형별 데이터 추출
di = train[train['시술 유형'] == 'DI']
ivf = train[train['시술 유형'] == 'IVF']

# 각 시술 유형의 컬럼별 결측률 계산
comparison = pd.DataFrame({
    'DI 결측률 (%)': di.isnull().mean() * 100,
    'IVF 결측률 (%)': ivf.isnull().mean() * 100
})

# DI 결측률과 IVF 결측률 차이 계산
comparison['결측률 차이 (%p)'] = (
    comparison['DI 결측률 (%)']
    - comparison['IVF 결측률 (%)']
)

# DI에서 결측치가 존재하는 컬럼만 선택
comparison = comparison[
    comparison['DI 결측률 (%)'] > 0
]

# DI 결측률이 높은 순서로 정렬
comparison = comparison.sort_values(
    by='DI 결측률 (%)',
    ascending=False
)

# 전체 결과 출력
display(comparison.round(2))


## ==========================================

원본 코드 셀 23. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# ==========================================
# 수치형 변수 선택
# ==========================================

# 0/1 변수와 타깃 제외
num_cols = [
    col for col in train.select_dtypes(include='number').columns
    if col != '임신 성공 여부'
    and train[col].nunique(dropna=True) > 2
]

results = []

# ==========================================
# Train 이상치 탐지
# ==========================================
# 대회 규칙에 따라 Test 데이터는 분석하지 않고 Train 데이터만 분석합니다.

for col in num_cols:

    # 결측치 제외
    train_s = train[col].dropna()

    # IQR 계산
    q1 = train_s.quantile(0.25)
    q3 = train_s.quantile(0.75)
    iqr = q3 - q1

    # IQR=0이면 이상치 판정 보류
    # 정상적인 값까지 이상치로 판단할 가능성이 높기 때문
    if pd.isna(iqr) or iqr == 0:
        train_count = np.nan
        train_ratio = np.nan

    else:
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        # 이상치 개수와 비율 계산 (결측치 제외)
        train_count = ((train_s < lower) | (train_s > upper)).sum()
        train_ratio = train_count / len(train_s) * 100

    results.append([col, q1, q3, iqr, train_count, train_ratio])

# ==========================================
# 이상치 분석 결과 정리
# ==========================================

outlier_df = pd.DataFrame(
    results,
    columns=['변수', 'Q1', 'Q3', 'IQR', 'Train 이상치 수', 'Train 이상치 비율(%)']
)

# 이상치 비율 기준 내림차순
outlier_df = outlier_df.sort_values('Train 이상치 비율(%)', ascending=False)

display(outlier_df.round(2))


## 분석 단계 27

원본 코드 셀 27. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
sns.set_theme(style='whitegrid')
plt.rcParams['font.family'] = 'Malgun Gothic' if platform.system() == 'Windows' else 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False


## 분석 단계 28

원본 코드 셀 28. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
plt.figure(figsize=(12, 7))

sns.barplot(
    data=outlier_df,
    x='Train 이상치 비율(%)',
    y='변수',
    color='steelblue'
)

plt.title('주요 수치형 변수별 이상치 비율', fontsize=15)
plt.xlabel('이상치 비율 (%)')
plt.ylabel('변수')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


## 이상치 비율이 높은 상위 8개 변수 선택

원본 코드 셀 29. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# 이상치 비율이 높은 상위 8개 변수 선택
top_cols = outlier_df[
    outlier_df['Train 이상치 수'] > 0
].head(8)['변수'].tolist()

# 4행 2열 그래프 생성
fig, axes = plt.subplots(4, 2, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(top_cols):

    # 결측치를 제외한 수치형 데이터
    values = train[col].dropna()

    # IQR 계산
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    # 이상치 추출
    outliers = values[
        (values < lower) | (values > upper)
    ]

    # 전체 데이터와 이상치의 구간을 동일하게 설정
    bins = np.histogram_bin_edges(values, bins=30)

    # 전체 데이터 분포
    sns.histplot(
        values,
        bins=bins,
        color='steelblue',
        alpha=0.6,
        ax=axes[i],
        label='전체 데이터',
        common_norm=False,
        element='step',
        fill=True,
        kde=True,
        line_kws={'linewidth': 1.8}
    )

    # 이상치 분포 강조
    sns.histplot(
        outliers,
        bins=bins,
        color='tomato',
        alpha=0.8,
        ax=axes[i],
        label='이상치',
        common_norm=False,
        element='step',
        fill=True,
        kde=True,
        line_kws={'linewidth': 1.8}
    )

    # IQR 상한선 표시
    axes[i].axvline(
        upper,
        color='red',
        linestyle='--',
        label='IQR 상한'
    )

    # IQR 하한선 표시
    axes[i].axvline(
        lower,
        color='green',
        linestyle='--',
        label='IQR 하한'
    )

    axes[i].set_title(col, fontsize=13)
    axes[i].set_xlabel('관측값')
    axes[i].set_ylabel('빈도')
    axes[i].legend()
    axes[i].grid(alpha=0.2)

# 사용하지 않는 그래프 영역 제거
for j in range(len(top_cols), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('주요 수치형 변수의 이상치 분포', fontsize=16)
plt.tight_layout()
plt.show()


## ==========================================

원본 코드 셀 34. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# ==========================================
# 입력 변수(X)와 타깃(y) 분리
# ==========================================
# 시간적 누수가 없다는 증거가 없는 임신·출산 누적 이력은 보수적으로 제외합니다.
# 데이터 명세/원천 시스템에서 '현재 시술 이전 이력'임이 확인되면 별도 실험으로 재검토합니다.
HISTORY_RISK_COLS = ['총 임신 횟수', 'IVF 임신 횟수', 'DI 임신 횟수',
                     '총 출산 횟수', 'IVF 출산 횟수', 'DI 출산 횟수']
assert set(HISTORY_RISK_COLS).issubset(train.columns)
assert set(HISTORY_RISK_COLS).issubset(test.columns)
X_train = train.drop(columns=['ID', '임신 성공 여부'] + HISTORY_RISK_COLS).copy()
y_train = train['임신 성공 여부'].astype(int)
X_test = test.drop(columns=['ID'] + HISTORY_RISK_COLS).copy()
print('누수 위험 검토 전까지 제외된 열:', HISTORY_RISK_COLS)

# ==========================================
# 변수 유형 분류
# ==========================================
count_cols = [c for c in X_train.columns if c.endswith('횟수')]
cat_cols = [c for c in X_train.columns
            if not pd.api.types.is_numeric_dtype(X_train[c]) and c not in count_cols]
num_cols = [c for c in X_train.columns if c not in count_cols + cat_cols]
bool_cols = [c for c in num_cols
             if ('여부' in c or '불임 원인' in c) and X_train[c].dropna().isin([0, 1]).all()]
day_cols = [c for c in num_cols if c.endswith('경과일')]
other_num_cols = [c for c in num_cols if c not in bool_cols + day_cols]

# 파생 변수 대상: 이상치 분석에서 이상치 비율이 높게 나타난 상위 8개 변수
flag_cols = ['저장된 배아 수', '미세주입에서 생성된 배아 수',
             '임신 시도 또는 마지막 임신 경과 연수', '미세주입된 난자 수',
             '혼합된 난자 수', '총 생성 배아 수',
             '수집된 신선 난자 수', '파트너 정자와 혼합된 난자 수']

# 분류 결과 확인
assert len(count_cols + cat_cols + bool_cols + day_cols + other_num_cols) == X_train.shape[1]
display(pd.DataFrame({
    '변수 유형': ['횟수형', '범주형', '여부형', '경과일', '기타 수치형', '합계'],
    '변수 수': [len(count_cols), len(cat_cols), len(bool_cols),
               len(day_cols), len(other_num_cols), X_train.shape[1]]
}).set_index('변수 유형'))


## 횟수형: '3회' → 3, '6회 이상' → 6, 결측 → 0

원본 코드 셀 36. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
def iqr_bounds(X_fit):
    """학습 데이터로 IQR 이상치 하한·상한을 계산합니다. IQR이 0이면 판정을 보류(None)합니다."""
    bounds = {}
    for col in flag_cols:
        s = X_fit[col].dropna()
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        bounds[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr) if iqr > 0 else None
    return bounds


def fill_values(X):
    """변수 유형별 규칙에 따라 결측치를 대체합니다. 학습 데이터의 통계값을 사용하지 않습니다."""
    df = X.copy()

    # 횟수형: '3회' → 3, '6회 이상' → 6, 결측 → 0
    for col in count_cols:
        df[col] = (df[col].astype(str)
                   .str.extract(r'(\d+)', expand=False)
                   .astype(float).fillna(0))

    # 여부형·경과일: 실제 0과 구분하기 위해 -1로 대체
    df[bool_cols] = df[bool_cols].fillna(-1)
    df[day_cols] = df[day_cols].fillna(-1)

    # 기타 수치형: 0으로 대체
    df[other_num_cols] = df[other_num_cols].fillna(0)

    # DI 시술의 구조적 결측은 '해당 없음', 그 외 범주형 결측은 'missing'
    is_di = df['시술 유형'].eq('DI') & df['배아 생성 주요 이유'].isna()
    df.loc[is_di, '배아 생성 주요 이유'] = '해당 없음'
    df[cat_cols] = df[cat_cols].fillna('missing').astype(str)
    return df


def prepare(X_fit, X, mode='baseline'):
    """선택한 mode에 따라 전처리를 수행하고 모델 입력 데이터를 반환합니다."""
    if mode not in ('baseline', 'flags', 'flags_iqr'):
        raise ValueError('mode는 baseline, flags, flags_iqr 중 하나여야 합니다.')

    df = fill_values(X)

    # ② 실험 A: 결측 여부 / 0 여부 플래그 (대체 이전 원본 값 기준)
    if mode in ('flags', 'flags_iqr'):
        for col in flag_cols:
            df[col + '_결측여부'] = X[col].isna().astype(int)
            df[col + '_0여부'] = X[col].eq(0).astype(int)

    # ③ 실험 B: IQR 이상치 여부 플래그 (경계값은 학습 데이터에서만 계산)
    if mode == 'flags_iqr':
        for col, b in iqr_bounds(X_fit).items():
            df[col + '_IQR이상치'] = (0 if b is None
                                      else ((X[col] < b[0]) | (X[col] > b[1])).astype(int))

    # 범주형 인코딩: 학습 데이터의 범주 사전만 사용하며, 처음 등장한 범주는 -1로 처리합니다.
    fit_cats = fill_values(X_fit)[cat_cols]
    for col in cat_cols:
        categories = sorted(fit_cats[col].unique())
        df[col] = pd.Categorical(df[col], categories=categories).codes

    if df.isna().any().any():
        raise ValueError('전처리 후 결측치가 남아 있습니다.')
    return df


def make_model():
    """모든 실험에 동일하게 사용하는 XGBoost 분류 모델을 생성합니다."""
    return XGBClassifier(
        n_estimators=300,        # 트리 개수
        max_depth=4,             # 트리 최대 깊이 (과적합 억제)
        learning_rate=0.05,      # 학습률
        tree_method='hist',      # 대용량 데이터용 히스토그램 방식
        eval_metric='auc',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )


# 전처리 결과 검증
check = []
for m in ['baseline', 'flags', 'flags_iqr']:
    sample = prepare(X_train, X_train.head(1000), m)
    check.append({'모드': m, '변수 수': sample.shape[1], '남은 결측치': int(sample.isna().sum().sum())})
display(pd.DataFrame(check).set_index('모드'))


## ==========================================

원본 코드 셀 38. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# ==========================================
# ④ 실험 C: 임상 지식 기반 결측 처리 + 결측 패턴 특성 공학
# ==========================================
# 다른 행의 통계는 사용하지 않지만 시술 정보의 예측 시점 적합성은 별도로 검토해야 합니다.

DAY_COLS = ['난자 채취 경과일', '난자 해동 경과일', '난자 혼합 경과일', '배아 이식 경과일', '배아 해동 경과일']

# 연령 구간 → 대표값(세). 환자 나이와 기증자 나이를 같은 척도로 비교하기 위함
PATIENT_AGE_YEARS = {'만18-34세': 30, '만35-37세': 36, '만38-39세': 38.5, '만40-42세': 41,
                     '만43-44세': 43.5, '만45-50세': 47.5}
DONOR_AGE_YEARS = {'만20세 이하': 20, '만21-25세': 23, '만26-30세': 28, '만31-35세': 33,
                   '만36-40세': 38, '만41-45세': 43}

# 결측이 '검사·시술을 하지 않음'을 의미하는 변수 (값이 1 또는 결측만 존재)
NOT_PERFORMED_COLS = ['착상 전 유전 검사 사용 여부', 'PGD 시술 여부', 'PGS 시술 여부']


def _num(s):
    return pd.to_numeric(s, errors='coerce')


def _count(s):
    return s.astype(str).str.extract(r'(\d+)', expand=False).astype(float)


def _ratio(a, b):
    b = b.astype(float)
    return (a / b.where(b > 0)).fillna(-1)


def clinical_features(X):
    """원본 X(결측 대체 전)로 임상 파생 변수와 결측 패턴 변수를 만듭니다."""
    f = pd.DataFrame(index=X.index)
    is_di = X['시술 유형'].eq('DI')
    n_transfer = X['이식된 배아 수']

    # ---------- 1) 결측 패턴 특성 ----------
    # 경과일 5개의 결측 여부를 비트로 묶은 코드 (예: 채취·혼합 결측 = 동결 배아 주기)
    f['경과일_결측패턴'] = sum(X[c].isna().astype(int) * (2 ** i) for i, c in enumerate(DAY_COLS))
    f['경과일_결측수'] = X[DAY_COLS].isna().sum(axis=1)
    f['전체_결측수'] = X.isna().sum(axis=1)
    # IVF인데 이식 경과일이 없음 = 이식이 이루어지지 않은 주기 (전량 동결·취소)
    f['IVF_이식일결측'] = (~is_di & X['배아 이식 경과일'].isna()).astype(int)
    # 동결 배아 주기인데 해동 경과일이 없음 / 신선 주기인데 채취일이 없음 = 기록 불일치
    f['동결주기_해동일결측'] = (X['동결 배아 사용 여부'].eq(1) & X['배아 해동 경과일'].isna()).astype(int)
    f['신선주기_채취일결측'] = (X['신선 배아 사용 여부'].eq(1) & X['난자 채취 경과일'].isna()).astype(int)

    # ---------- 2) 시술 주기 유형 (임상 분류) ----------
    f['주기_DI'] = is_di.astype(int)
    f['주기_이식없음'] = (~is_di & n_transfer.fillna(0).eq(0)).astype(int)        # 성공률 ≈ 0
    f['주기_동결배아이식'] = (~is_di & X['동결 배아 사용 여부'].eq(1)).astype(int)  # FET
    f['주기_신선배아이식'] = (~is_di & X['신선 배아 사용 여부'].eq(1)).astype(int)
    f['주기_기증난자'] = X['난자 출처'].eq('기증 제공').astype(int)
    f['주기_기증배아'] = X['기증 배아 사용 여부'].eq(1).astype(int)
    f['주기_저장목적'] = (X['배아 생성 주요 이유'].fillna('').str.contains('저장|기증|연구')
                       & ~X['배아 생성 주요 이유'].fillna('').str.contains('현재 시술용')).astype(int)

    # ---------- 3) 배아 발달 단계 (Day 3 vs Day 5 배반포) ----------
    culture = X['배아 이식 경과일'] - X['난자 혼합 경과일']
    f['배양일수'] = culture.fillna(-1)
    f['배반포이식'] = ((culture >= 5) | X['특정 시술 유형'].fillna('').str.contains('BLASTOCYST')).astype(int)
    f['해동후이식일'] = (X['배아 이식 경과일'] - X['배아 해동 경과일']).fillna(-1)

    # ---------- 4) 실제 난자 나이 ----------
    # 임상적으로 성공률은 '난자 제공자'의 나이에 좌우됩니다. 기증 난자면 기증자 나이를 사용합니다.
    patient_age = X['시술 당시 나이'].map(PATIENT_AGE_YEARS)
    donor_age = X['난자 기증자 나이'].map(DONOR_AGE_YEARS)
    egg_age = np.where(f['주기_기증난자'].eq(1) & donor_age.notna(), donor_age, patient_age)
    f['환자나이_세'] = patient_age.fillna(-1)
    f['난자나이_세'] = pd.Series(egg_age, index=X.index).fillna(-1)
    f['고령_본인난자'] = ((patient_age >= 38) & f['주기_기증난자'].eq(0)).astype(int)

    # ---------- 5) 난자·배아 지표 ----------
    f['수정률'] = _ratio(X['총 생성 배아 수'], X['혼합된 난자 수'])
    f['ICSI수정률'] = _ratio(X['미세주입에서 생성된 배아 수'], X['미세주입된 난자 수'])
    f['배아생성률_채취대비'] = _ratio(X['총 생성 배아 수'], X['수집된 신선 난자 수'])
    f['이식비율'] = _ratio(n_transfer, X['총 생성 배아 수'])
    f['저장비율'] = _ratio(X['저장된 배아 수'], X['총 생성 배아 수'])
    f['잉여배아_저장'] = (X['저장된 배아 수'].fillna(0) > 0).astype(int)  # 양질 배아가 남았다는 신호
    f['총난자수'] = X['수집된 신선 난자 수'].fillna(0) + X['해동 난자 수'].fillna(0)

    # ---------- 6) 과거 시술·임신 이력 ----------
    total = _count(X['총 시술 횟수'])
    # 임신·출산 횟수 6개는 실제 현재 시술 이전 이력인지 불명확하여 사용하지 않습니다.
    f['타클리닉시술'] = (total - _count(X['클리닉 내 총 시술 횟수'])).fillna(-1)

    # ---------- 7) 불임 원인 개수 ----------
    cause = [c for c in X.columns if '불임 원인' in c]
    f['불임원인_개수'] = X[cause].fillna(0).sum(axis=1)
    return f


if '_prepare_base' not in globals():   # 셀을 여러 번 실행해도 중복 래핑되지 않도록
    _prepare_base = prepare


def prepare(X_fit, X, mode='baseline'):
    """mode='clinical'이면 실험 A 전처리에 임상 결측 처리와 임상·결측 패턴 변수를 더합니다."""
    if mode != 'clinical':
        return _prepare_base(X_fit, X, mode)
    df = _prepare_base(X_fit, X, 'flags')
    # 유전 검사 결측=미시행을 단정할 근거가 없어 결측(-1)을 유지합니다.
    for col in NOT_PERFORMED_COLS:
        df[col] = X[col].fillna(-1)
    df = pd.concat([df, clinical_features(X)], axis=1)
    if df.isna().any().any():
        raise ValueError('전처리 후 결측치가 남아 있습니다.')
    return df


sample = prepare(X_train, X_train.head(1000), 'clinical')
print(f"clinical 모드 변수 수: {sample.shape[1]} (실험 A 대비 +{sample.shape[1] - prepare(X_train, X_train.head(10), 'flags').shape[1]}개), "
      f"남은 결측치: {int(sample.isna().sum().sum())}")


## 원본 Test 대신 학습 데이터에서 분리한 표본으로 행 단위 전처리 독립성을 검사합니다.

원본 코드 셀 40. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# 원본 Test 대신 학습 데이터에서 분리한 표본으로 행 단위 전처리 독립성을 검사합니다.
# 자동 점검은 시간적 누수, 동일 환자 중복, 타깃 누수 부재를 보장하지 않습니다.
def check_leakage_fast(mode):
    fit_X = X_train.sample(n=min(10000, len(X_train)//2), random_state=42)
    probe_X = X_train.drop(index=fit_X.index).sample(n=min(1000, len(X_train)-len(fit_X)), random_state=43)
    full = prepare(fit_X, probe_X, mode)
    chunks = pd.concat([prepare(fit_X, probe_X.iloc[i:i+250], mode)
                        for i in range(0, len(probe_X), 250)])
    reordered = probe_X.sample(frac=1, random_state=42)
    train_prep = prepare(fit_X, fit_X, mode)
    return {
        '① 행 독립성': full.equals(chunks),
        '② 순서 독립성': full.sort_index().equals(prepare(fit_X, reordered, mode).sort_index()),
        '③ 범주 코드 범위': all(full[c].between(-1, train_prep[c].max()).all() for c in cat_cols),
        '④ 후속 호출 결과 불변': full.equals(prepare(fit_X, probe_X, mode)),
    }

leak_df = pd.DataFrame({m: check_leakage_fast(m)
                        for m in ['baseline', 'flags', 'flags_iqr', 'clinical']})
display(leak_df.replace({True: '✅ 통과', False: '❌ 실패'}))
assert leak_df.values.all(), '전처리 독립성 점검 실패'
print('표본 기반 전처리 독립성 점검 통과 — 시간적 누수는 별도 검토 필요')


## 분석 단계 42

원본 코드 셀 42. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

display(pd.DataFrame({
    '행 수': [len(X_tr), len(X_val), len(X_test)],
    '임신 성공 비율(%)': [y_tr.mean() * 100, y_val.mean() * 100, np.nan]
}, index=['학습 데이터', '홀드아웃 검증 데이터', 'Test 데이터']).round(2))


## 전처리 기준은 학습 Fold(X_a)에서만 계산합니다.

원본 코드 셀 44. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
if not ALLOW_TRAINING:
    raise RuntimeError("교차 검증/학습은 ALLOW_TRAINING=True일 때만 실행합니다.")
EXPERIMENTS = {
    '① Baseline': 'baseline',
    '② 실험 A (결측·0 플래그)': 'flags',
    '③ 실험 B (결측·0·IQR 플래그)': 'flags_iqr',
    '④ 실험 C (임상 결측 처리·결측 패턴)': 'clinical',
}

records = []
for name, mode in EXPERIMENTS.items():
    train_scores, cv_scores = [], []
    for tr_idx, va_idx in cv.split(X_tr, y_tr):
        X_a, X_b = X_tr.iloc[tr_idx], X_tr.iloc[va_idx]
        y_a, y_b = y_tr.iloc[tr_idx], y_tr.iloc[va_idx]

        # 전처리 기준은 학습 Fold(X_a)에서만 계산합니다.
        X_a_prep = prepare(X_a, X_a, mode)
        X_b_prep = prepare(X_a, X_b, mode)

        model = make_model()
        model.fit(X_a_prep, y_a)
        train_scores.append(roc_auc_score(y_a, model.predict_proba(X_a_prep)[:, 1]))
        cv_scores.append(roc_auc_score(y_b, model.predict_proba(X_b_prep)[:, 1]))

    print(f'{name} 검증 완료 | CV ROC-AUC {np.mean(cv_scores):.5f}')
    records.append({'실험': name, '모드': mode,
                    '추가 변수 수': prepare(X_tr, X_tr.head(10), mode).shape[1] - X_tr.shape[1],
                    'Train ROC-AUC': np.mean(train_scores),
                    'CV ROC-AUC 평균': np.mean(cv_scores),
                    'CV ROC-AUC 표준편차': np.std(cv_scores)})

result_df = pd.DataFrame(records)
baseline_auc = result_df.loc[result_df['모드'] == 'baseline', 'CV ROC-AUC 평균'].iloc[0]
result_df['Baseline 대비 변화'] = result_df['CV ROC-AUC 평균'] - baseline_auc
result_df = result_df.sort_values('CV ROC-AUC 평균', ascending=False).reset_index(drop=True)
display(result_df.set_index('실험').round(5))

best_mode = result_df.loc[0, '모드']
best_name = result_df.loc[0, '실험']
print(f'\n선정된 전처리 방식: {best_name}')


## 분석 단계 45

원본 코드 셀 45. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
chart = result_df.sort_values('CV ROC-AUC 평균')
plt.figure(figsize=(10, 4))
bars = plt.barh(chart['실험'], chart['CV ROC-AUC 평균'], xerr=chart['CV ROC-AUC 표준편차'],
                color=['#9db4cf' if m != best_mode else '#2f5d8a' for m in chart['모드']], capsize=5)
for bar, v in zip(bars, chart['CV ROC-AUC 평균']):
    plt.text(v + 0.0045, bar.get_y() + bar.get_height() / 2, f'{v:.5f}', va='center')
plt.axvline(baseline_auc, color='tomato', linestyle='--', label=f'Baseline {baseline_auc:.4f}')
plt.xlim(0.70, 0.76)
plt.xlabel('ROC-AUC (5-Fold 평균 ± 표준편차)')
plt.title('전처리 방식별 교차 검증 ROC-AUC 비교', fontsize=14)
plt.legend(loc='lower right')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


## 실험 설정: 전체 데이터 실행 시 아래 값을 사용합니다.

원본 코드 셀 48. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
import time, gc, json, sys, warnings
from datetime import datetime, timezone, timedelta
from catboost import CatBoostClassifier
from sklearn.metrics import roc_curve, confusion_matrix, ConfusionMatrixDisplay
import xgboost, catboost, sklearn

# 실험 설정: 전체 데이터 실행 시 아래 값을 사용합니다.
MODEL_FOLDS = 5
XGB_SEEDS = [42, 2024, 7]    # 기존 v3: 3시드 평균
MODEL_THREADS = 2            # Colab CPU 과다 사용 방지
CAT_INNER_VALID = 0.12       # 학습 Fold 내부에서만 iterations 선택
CAT_MAX_ITERS = 1300
CAT_EARLY_STOP = 100
SMOKE_TEST = False           # 빠른 코드 시험 시 True; 제출용에서는 반드시 False

# v3 Optuna 출력(반올림값): 정확한 소수 자릿수는 원본 Optuna DB를 확인해야 함
XGB_V3_PARAMS = dict(
    max_depth=5, min_child_weight=1.8533, subsample=0.7095,
    colsample_bytree=0.5675, gamma=0.0018, reg_alpha=0.0153,
    reg_lambda=68.4459, learning_rate=0.05,
)
XGB_TREES = 345

# CatBoost 고정 초기 파라미터 (성능 최적임을 보장하지 않는 사전 지정값)
CAT_PARAMS = dict(depth=5, learning_rate=0.035, l2_leaf_reg=5.,
                  random_strength=0.8, loss_function='Logloss', eval_metric='AUC')

assert set(HISTORY_RISK_COLS).isdisjoint(X_tr.columns)
assert set(HISTORY_RISK_COLS).isdisjoint(X_test.columns)
assert X_tr.columns.tolist() == X_val.columns.tolist() == X_test.columns.tolist()

OUT_DIR = PROJECT_PATH / 'ensemble_xgb_catboost_v3'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('전처리 모드:', best_mode)
print('XGB Seed:', XGB_SEEDS, '| XGB 나무 수:', XGB_TREES)
print('CatBoost 상한:', CAT_MAX_ITERS, '| 내부 early stopping:', CAT_EARLY_STOP)
print('출력 위치:', OUT_DIR)


## v3의 clinical_features() / fill_values() / flag_cols / cat_cols를 그대로 사용

원본 코드 셀 50. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# v3의 clinical_features() / fill_values() / flag_cols / cat_cols를 그대로 사용
# 차이점: CatBoost 범주형은 원본 문자열이므로 ordinal encoding을 하지 않습니다.

def prepare_catboost(X_raw, mode):
    if mode not in ('baseline', 'flags', 'flags_iqr', 'clinical'):
        raise ValueError('잘못된 전처리 모드')
    z = fill_values(X_raw)  # 원본 v3의 고정 결측 처리
    if mode in ('flags', 'flags_iqr', 'clinical'):
        for column in flag_cols:
            z[column + '_결측여부'] = X_raw[column].isna().astype(int)
            z[column + '_0여부'] = X_raw[column].eq(0).astype(int)
    if mode == 'flags_iqr':
        # IQR은 CatBoost에서도 Fold의 X_fit에서만 계산해야 합니다.
        # 이 모드의 IQR 처리는 prepare_catboost_fold()에서 처리합니다.
        raise RuntimeError('flags_iqr는 prepare_catboost_fold(X_fit, X_new, mode)를 사용하세요')
    if mode == 'clinical':
        for column in NOT_PERFORMED_COLS:
            z[column] = X_raw[column].fillna(-1)
        z = pd.concat([z, clinical_features(X_raw)], axis=1)
    for column in cat_cols:
        z[column] = z[column].astype(str)
    assert not z.columns.duplicated().any()
    assert not z.isna().any().any()
    return z


def prepare_catboost_fold(X_fit, X_new, mode):
    if mode == 'flags_iqr':
        z = prepare_catboost(X_new, 'flags')
        for col,b in iqr_bounds(X_fit).items():
            z[col+'_IQR이상치'] = (0 if b is None else ((X_new[col]<b[0]) | (X_new[col]>b[1])).astype(int))
        return z
    return prepare_catboost(X_new, mode)


def make_xgb(seed):
    return XGBClassifier(
        **XGB_V3_PARAMS, n_estimators=XGB_TREES,
        tree_method='hist', objective='binary:logistic',
        eval_metric='auc', random_state=int(seed), n_jobs=MODEL_THREADS,
    )


def make_catboost(iterations, seed):
    return CatBoostClassifier(
        **CAT_PARAMS, iterations=int(iterations), random_seed=int(seed),
        thread_count=MODEL_THREADS, verbose=False,
        allow_writing_files=False,
    )

# 작은 표본에서 변수 정합성 확인
check_x_raw = X_tr.head(60)
for mode in ('baseline','flags','flags_iqr','clinical'):
    xb = prepare(check_x_raw, check_x_raw, mode)
    cb = prepare_catboost_fold(check_x_raw, check_x_raw, mode)
    assert xb.columns.tolist() == cb.columns.tolist(), (mode, 'XGB/CB 변수 구성 불일치')
    assert xb.shape == cb.shape
    print(f'{mode:<10} : 입력 변수 {xb.shape[1]}개, 결측 0 확인')


## 전체 실행 전 소규모 문법/배열/학습 동작 확인용 옵션

원본 코드 셀 52. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
if SMOKE_TEST:
    # 전체 실행 전 소규모 문법/배열/학습 동작 확인용 옵션
    X_dev, _, y_dev, _ = train_test_split(X_tr, y_tr, train_size=1200, stratify=y_tr, random_state=42)
    X_hold = X_val.iloc[:300].copy()
    y_hold = y_val.iloc[:300].copy()
    MODEL_FOLDS = 2
    XGB_SEEDS = [42]
    XGB_TREES = 30
    CAT_MAX_ITERS = 55
    CAT_EARLY_STOP = 12
    print('SMOKE_TEST=True: 제출 저장 없이 소규모 실행')
else:
    X_dev, y_dev = X_tr, y_tr
    X_hold, y_hold = X_val, y_val

cv_model = StratifiedKFold(n_splits=MODEL_FOLDS, shuffle=True, random_state=RANDOM_STATE)
oof_xgb = np.full(len(X_dev), np.nan)
oof_cb = np.full(len(X_dev), np.nan)
fold_rows, cat_iters = [], []
holdout_x_preds, holdout_c_preds = [], []
t_start = time.time()

for fold, (tr_idx, va_idx) in enumerate(cv_model.split(X_dev, y_dev), 1):
    Xa, Xb = X_dev.iloc[tr_idx], X_dev.iloc[va_idx]
    ya, yb = y_dev.iloc[tr_idx], y_dev.iloc[va_idx]
    # XGBoost : Train Fold 범주 사전만 적용
    A_x = prepare(Xa, Xa, best_mode)
    B_x = prepare(Xa, Xb, best_mode)
    H_x = prepare(Xa, X_hold, best_mode)
    xgb_fold_models = []
    for seed in XGB_SEEDS:
        xm = make_xgb(seed)
        xm.fit(A_x, ya, verbose=False)
        xgb_fold_models.append(xm)
    px = np.mean([m.predict_proba(B_x)[:,1] for m in xgb_fold_models], axis=0)
    hold_x = np.mean([m.predict_proba(H_x)[:,1] for m in xgb_fold_models], axis=0)

    # CatBoost 반복 수: 외부 학습 Fold의 내부 분할에서만 선택
    inner_a, inner_v = train_test_split(
        np.arange(len(Xa)), test_size=CAT_INNER_VALID,
        stratify=ya, random_state=RANDOM_STATE+fold,
    )
    Ai = prepare_catboost_fold(Xa.iloc[inner_a], Xa.iloc[inner_a], best_mode)
    Vi = prepare_catboost_fold(Xa.iloc[inner_a], Xa.iloc[inner_v], best_mode)
    probe = make_catboost(CAT_MAX_ITERS, seed=RANDOM_STATE+fold)
    probe.fit(Ai, ya.iloc[inner_a], eval_set=(Vi, ya.iloc[inner_v]),
              cat_features=cat_cols, early_stopping_rounds=CAT_EARLY_STOP,
              use_best_model=True)
    best_iter = max(1, int(probe.get_best_iteration())+1)
    cat_iters.append(best_iter)
    del probe, Ai, Vi
    # 정해진 나무 수로 외부 Fold의 전체 학습 데이터에 재학습
    A_c = prepare_catboost_fold(Xa, Xa, best_mode)
    B_c = prepare_catboost_fold(Xa, Xb, best_mode)
    H_c = prepare_catboost_fold(Xa, X_hold, best_mode)
    cm = make_catboost(best_iter, seed=RANDOM_STATE+fold)
    cm.fit(A_c, ya, cat_features=cat_cols)
    pc = cm.predict_proba(B_c)[:, 1]
    hold_c = cm.predict_proba(H_c)[:, 1]

    oof_xgb[va_idx], oof_cb[va_idx] = px, pc
    holdout_x_preds.append(hold_x)
    holdout_c_preds.append(hold_c)
    row = {
        'Fold': fold,
        'XGBoost': roc_auc_score(yb, px),
        'CatBoost': roc_auc_score(yb, pc),
        '50:50 앙상블': roc_auc_score(yb, (px + pc)/2),
        'CatBoost iterations': best_iter,
    }
    fold_rows.append(row)
    print(f'Fold {fold}/{MODEL_FOLDS} | XGB {row["XGBoost"]:.6f} | '
          f'CatBoost {row["CatBoost"]:.6f} | 50:50 {row["50:50 앙상블"]:.6f} '
          f'| CatBoost trees {best_iter} | {(time.time()-t_start)/60:.1f}분')
    del A_x, B_x, H_x, A_c, B_c, H_c, xgb_fold_models, cm
    gc.collect()

assert np.isfinite(oof_xgb).all() and np.isfinite(oof_cb).all()
fold_results = pd.DataFrame(fold_rows).set_index('Fold')
metric_cols = ['XGBoost', 'CatBoost', '50:50 앙상블']
cv_results = fold_results[metric_cols].agg(['mean', 'std']).T
cv_results['OOF ROC-AUC'] = [roc_auc_score(y_dev, oof_xgb),
                              roc_auc_score(y_dev, oof_cb),
                              roc_auc_score(y_dev, (oof_xgb+oof_cb)/2)]
print('\n동일 Fold에서 비교한 CV ROC-AUC:')
display(cv_results.round(6))
print('OOF 예측 순위 상관(Spearman):', round(pd.Series(oof_xgb).corr(pd.Series(oof_cb), method='spearman'), 5))
print('소요 시간:', round((time.time()-t_start)/60, 1), '분')


## 세 후보 CV 점수와 Fold별 차이 시각화

원본 코드 셀 54. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# 세 후보 CV 점수와 Fold별 차이 시각화
fig, axes = plt.subplots(1, 2, figsize=(14,5))
order = cv_results.sort_values('mean').index
axes[0].barh(order, cv_results.loc[order,'mean'], xerr=cv_results.loc[order,'std'], capsize=4)
axes[0].set_xlim(max(0.5, float(cv_results['mean'].min()-0.03)), min(1, float(cv_results['mean'].max()+0.025)))
axes[0].set_xlabel('ROC-AUC (평균 ± 표준편차)')
axes[0].set_title('동일 5-Fold CV ROC-AUC 비교')
axes[0].grid(axis='x', alpha=0.3)
for name in metric_cols:
    axes[1].plot(fold_results.index, fold_results[name], marker='o', label=name)
axes[1].set_xlabel('Fold'); axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('Fold별 모델 ROC-AUC')
axes[1].legend();axes[1].grid(alpha=0.3)
plt.tight_layout();plt.show()

BEST_MODEL = cv_results['mean'].idxmax()   # 홀드아웃 확인 이전에 고정
best_single = cv_results.loc[['XGBoost','CatBoost'],'mean'].idxmax()
blend_gain = float(cv_results.loc['50:50 앙상블','mean']-cv_results.loc[best_single,'mean'])
print('CV 최고 후보:', BEST_MODEL)
print(f'50:50 - 우수 단일 모델 CV 평균 차이: {blend_gain:+.6f}')
print('CV 점수만으로 선택 완료. 홀드아웃은 이제 처음 확인합니다.')


## 분석 단계 56

원본 코드 셀 56. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
hold_p_x = np.mean(np.stack(holdout_x_preds), axis=0)
hold_p_c = np.mean(np.stack(holdout_c_preds), axis=0)
hold_pred = {'XGBoost':hold_p_x,
             'CatBoost':hold_p_c,
             '50:50 앙상블':0.5*(hold_p_x+hold_p_c)}
holdout_scores = pd.DataFrame({name: {
    'ROC-AUC':roc_auc_score(y_hold,prob),
    'Accuracy':accuracy_score(y_hold, prob>=0.5),
    'Precision':precision_score(y_hold, prob>=0.5,zero_division=0),
    'Recall':recall_score(y_hold, prob>=0.5,zero_division=0),
    'F1-score':f1_score(y_hold, prob>=0.5,zero_division=0)
} for name,prob in hold_pred.items()}).T
display(holdout_scores.round(6))
print('CV로 선택된 모델:', BEST_MODEL, '| 해당 모델의 홀드아웃 ROC-AUC:', round(holdout_scores.loc[BEST_MODEL, 'ROC-AUC'],6))

plt.figure(figsize=(7,6))
for name,prob in hold_pred.items():
    fpr,tpr,_=roc_curve(y_hold,prob)
    plt.plot(fpr,tpr,label=f'{name} (AUC {roc_auc_score(y_hold,prob):.5f})')
plt.plot([0,1],[0,1],'--',color='gray')
plt.xlabel('거짓 양성 비율 (FPR)');plt.ylabel('참 양성 비율 (TPR)')
plt.title('홀드아웃 ROC 곡선: XGBoost / CatBoost / 50:50')
plt.legend(loc='lower right');plt.grid(alpha=0.3);plt.tight_layout();plt.show()


## 분석 단계 58

원본 코드 셀 58. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
from sklearn.metrics import roc_curve, classification_report
final_tr = prepare(X_tr, X_tr, best_mode)
final_val = prepare(X_tr, X_val, best_mode)
model = make_model()  # 원본 v3의 XGBoost 300 trees
model.fit(final_tr,y_tr)
train_prob = model.predict_proba(final_tr)[:,1]
val_prob = model.predict_proba(final_val)[:,1]
val_auc = roc_auc_score(y_val,val_prob)
print('원본 v3 기준 그래프를 위한 기본 XGBoost 홀드아웃 ROC-AUC:',round(val_auc,6))


## ROC 곡선

원본 코드 셀 59. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
# ROC 곡선
fpr, tpr, _ = roc_curve(y_val, val_prob)
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color='#2f5d8a', linewidth=2, label=f'XGBoost (AUC = {val_auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='무작위 예측 (AUC = 0.5)')
plt.fill_between(fpr, tpr, alpha=0.1, color='#2f5d8a')
plt.xlabel('거짓 양성 비율 (FPR)')
plt.ylabel('참 양성 비율 (TPR)')
plt.title('홀드아웃 검증 데이터 ROC 곡선', fontsize=14)
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 분석 단계 60

원본 코드 셀 60. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
importance = (pd.Series(model.get_booster().get_score(importance_type='gain'))
              .reindex(final_tr.columns).fillna(0)
              .sort_values(ascending=False))
top15 = importance.head(15)[::-1]

plt.figure(figsize=(10, 6))
plt.barh(top15.index, top15.values,
         color=['#e07a5f' if any(k in c for k in ['_결측여부', '_0여부', '_IQR이상치', '결측', '주기_', '배양일수', '배반포', '난자나이', '환자나이', '수정률', '비율', '잉여배아', '과거', '반복실패', '타클리닉', '불임원인_개수', '총난자수', '해동후이식일', '배아생성률', '고령']) else '#2f5d8a'
                for c in top15.index])
plt.xlabel('중요도 (gain)')
plt.title('변수 중요도 상위 15개 (주황색: 파생 변수)', fontsize=14)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

derived = [c for c in importance.index if any(k in c for k in ['_결측여부', '_0여부', '_IQR이상치', '결측', '주기_', '배양일수', '배반포', '난자나이', '환자나이', '수정률', '비율', '잉여배아', '과거', '반복실패', '타클리닉', '불임원인_개수', '총난자수', '해동후이식일', '배아생성률', '고령'])]
if derived:
    rank = {c: list(importance.index).index(c) + 1 for c in derived}
    best_derived = min(rank, key=rank.get)
    print(f'파생 변수 중 가장 중요도가 높은 변수: {best_derived} (전체 {len(importance)}개 중 {rank[best_derived]}위)')
else:
    print('선정된 전처리 방식에는 파생 변수가 포함되지 않습니다.')


## OOF와 CV Fold 점수, 선택 사유를 Train의 ID와 함께 저장

원본 코드 셀 62. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
if SMOKE_TEST:
    print('SMOKE_TEST=True: 제출 CSV 생성 생략')
else:
    cat_final_iterations = max(1, int(np.median(cat_iters)))
    full_cv = StratifiedKFold(n_splits=MODEL_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    tx_list, tc_list = [], []
    t_final = time.time()
    for fold,(tr_idx,_) in enumerate(full_cv.split(X_train,y_train),1):
        Xa,ya = X_train.iloc[tr_idx],y_train.iloc[tr_idx]
        Ax = prepare(Xa,Xa,best_mode)
        Tx = prepare(Xa,X_test,best_mode)
        xp = []
        for seed in XGB_SEEDS:
            m = make_xgb(seed)
            m.fit(Ax, ya, verbose=False)
            xp.append(m.predict_proba(Tx)[:,1])
            del m
        tx_list.append(np.mean(xp,axis=0))
        Ac = prepare_catboost_fold(Xa,Xa,best_mode)
        Tc = prepare_catboost_fold(Xa,X_test,best_mode)
        cm = make_catboost(cat_final_iterations,seed=RANDOM_STATE+fold)
        cm.fit(Ac,ya,cat_features=cat_cols)
        tc_list.append(cm.predict_proba(Tc)[:,1])
        print(f'전체 Train Fold {fold}/{MODEL_FOLDS} 완료 | {(time.time()-t_final)/60:.1f}분')
        del Ax,Tx,Ac,Tc,cm,xp
        gc.collect()

    test_x = np.mean(np.stack(tx_list),axis=0)
    test_c = np.mean(np.stack(tc_list),axis=0)
    test_blend = 0.5*test_x+0.5*test_c
    test_prob_candidates = {'XGBoost':test_x,'CatBoost':test_c,'50:50 앙상블':test_blend}
    fmt_names={'XGBoost':'xgboost','CatBoost':'catboost','50:50 앙상블':'blend5050'}

    sample_path = DATA_PATH / 'sample_submission.csv'
    sample_submission = pd.read_csv(sample_path,encoding='utf-8-sig')
    assert sample_submission.columns.tolist()==['ID','probability']
    assert len(sample_submission)==len(X_test)==len(test)
    assert sample_submission['ID'].equals(test['ID'])
    assert test['ID'].is_unique
    stamp = datetime.now(timezone(timedelta(hours=9))).strftime('%Y%m%d_%H%M%S')
    submission_paths={}
    for name,pred in test_prob_candidates.items():
        pred=np.asarray(pred,dtype=float)
        assert pred.shape==(len(test),)
        assert np.isfinite(pred).all()
        assert ((pred>=0)&(pred<=1)).all()
        sub=sample_submission.copy()
        sub['probability']=pred
        path=OUT_DIR / f'submission_v3_{fmt_names[name]}_{stamp}.csv'
        sub.to_csv(path,index=False,encoding='utf-8-sig')
        check_csv=pd.read_csv(path,encoding='utf-8-sig')
        assert check_csv.columns.tolist()==['ID','probability']
        assert check_csv['ID'].equals(test['ID'])
        assert check_csv['probability'].between(0,1).all()
        assert check_csv['probability'].notna().all()
        submission_paths[name]=str(path)
        print(f'{name:14} → {path.name} / {len(sub):,}행')

    # OOF와 CV Fold 점수, 선택 사유를 Train의 ID와 함께 저장
    oof_frame=pd.DataFrame({
        'ID':train.loc[X_dev.index,'ID'].to_numpy(),
        'target':y_dev.to_numpy(),
        'xgb_oof':oof_xgb,
        'catboost_oof':oof_cb,
        'blend5050_oof':(oof_xgb+oof_cb)/2,
    })
    oof_frame.to_csv(OUT_DIR/f'dev_oof_v3_{stamp}.csv',index=False,encoding='utf-8-sig')
    fold_results.to_csv(OUT_DIR/f'cv_fold_results_{stamp}.csv',encoding='utf-8-sig')
    cv_results.to_csv(OUT_DIR/f'cv_comparison_{stamp}.csv',encoding='utf-8-sig')
    report={
      'run_id':stamp, 'source':'XGBoost_v3_임상결측',
      'cv_selected':BEST_MODEL, 'cv_mean':cv_results['mean'].to_dict(),
      'cv_std':cv_results['std'].to_dict(),
      'holdout_auc_report_only':holdout_scores['ROC-AUC'].to_dict(),
      'history_excluded':HISTORY_RISK_COLS,'preprocessing_mode':best_mode,
      'xgb_params_from_rounded_v3_output':{**XGB_V3_PARAMS,'n_estimators':XGB_TREES},
      'xgb_seeds':XGB_SEEDS, 'cat_params':{**CAT_PARAMS,'iterations':cat_final_iterations},
      'cat_dev_fold_iterations':cat_iters,'files':submission_paths,
      'versions':{'python':sys.version.split()[0],
                  'sklearn':sklearn.__version__,
                  'xgboost':xgboost.__version__,
                  'catboost':catboost.__version__},
    }
    (OUT_DIR/f'experiment_v3_{stamp}.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
    print('CV 기준 추천 제출 파일:',submission_paths[BEST_MODEL])
